# Broad predictions (b0)

Loads the supplied e15 and e19 checkpoints and applies them to the b1 holdout cache with their saved BatchNorm statistics. Each model combines T5 cosine similarities with classification scores. Their row-standardised scores are averaged and converted to normalised ranks in submission_broad.csv. Downloads b1 and the checkpoints from Hugging Face.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path(".")
CACHE_DIR = None
WEIGHTS = None
CACHE_VERSION = "b1"
DEPTH = 16
NUM_WORKERS = 2

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pnpl==0.2.0", "x-transformers==2.31.10", "huggingface-hub"],
    check=True,
)

In [ ]:
from huggingface_hub import snapshot_download

if CACHE_DIR is None:
    CACHE_DIR = snapshot_download(
        repo_id="JatinArutla/PNPL-2026-B1-Cache",
        repo_type="dataset",
        allow_patterns=["meta_b1.npz", "hold_b1.bin", "t5_b1.bin"],
    )
if WEIGHTS is None:
    WEIGHTS = snapshot_download(
        repo_id="JatinArutla/PNPL-2026-E15-E19-Checkpoints",
        repo_type="dataset",
        allow_patterns=["best_e15.pt", "best_e19.pt"],
    )

CACHE_DIR = Path(CACHE_DIR)
WEIGHTS = Path(WEIGHTS)
CHECKPOINTS = {run: WEIGHTS / f"best_{run}.pt" for run in ("e15", "e19")}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for filename in ("meta_b1.npz", "hold_b1.bin", "t5_b1.bin"):
    if not (CACHE_DIR / filename).is_file():
        raise FileNotFoundError(f"Missing b1 input: {CACHE_DIR / filename}")
for path in CHECKPOINTS.values():
    if not path.is_file():
        raise FileNotFoundError(f"Missing checkpoint: {path}")

In [ ]:
import math
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from x_transformers import Encoder
from pnpl.competition import load_vocabulary, write_submission

device = "cuda"
assert torch.cuda.is_available(), "Select a T4 GPU."
autocast_settings = dict(device_type="cuda", dtype=torch.float16)
T0 = time.time()
metadata = np.load(CACHE_DIR / f"meta_{CACHE_VERSION}.npz", allow_pickle=True)
FREQ = float(metadata["freq"])
holdout_meg = np.memmap(
    CACHE_DIR / f"hold_{CACHE_VERSION}.bin", dtype=np.float16, mode="r"
).reshape(-1, 306)
word_embeddings = np.memmap(
    CACHE_DIR / f"t5_{CACHE_VERSION}.bin", dtype=np.float16, mode="r"
).reshape(-1, int(metadata["t5_dim"]))
sensor_positions = torch.tensor(metadata["positions"], dtype=torch.float32).to(device)
h_subject = metadata["hold_subject"].astype(np.int64)
assert 0 not in set(h_subject.tolist()), "subject 0 belongs to the deep track"
assert len(h_subject) == 37439
assert set(h_subject.tolist()) == set(range(1, 40))
assert str(metadata["version"]) == "b1"
assert FREQ == 50.0
print(f"cache {CACHE_DIR} | holdout {len(h_subject)} rows")

In [ ]:
class FourierEmb(nn.Module):
    def __init__(self, dimension=2048, margin=0.2):
        super().__init__()
        self.dimension = dimension
        self.margin = margin

    def forward(self, positions):
        leading_shape = positions.shape[:-1]
        count = int((self.dimension // 2) ** 0.5)
        frequencies = torch.arange(
            count, device=positions.device, dtype=positions.dtype
        )
        width = 1 + 2 * self.margin
        x_frequencies = 2 * math.pi * frequencies[:, None] / width
        y_frequencies = 2 * math.pi * frequencies / width
        shifted = (positions + self.margin)[..., None, None, :]
        angles = shifted[..., 0] * x_frequencies + shifted[..., 1] * y_frequencies
        angles = angles.reshape(*leading_shape, -1)
        return torch.cat([torch.cos(angles), torch.sin(angles)], dim=-1)


class ChannelMerger(nn.Module):
    def __init__(self, chout=270, pos_dim=2048, dropout=0.2):
        super().__init__()
        self.heads = nn.Parameter(torch.randn(1, chout, pos_dim) / pos_dim**0.5)
        self.embedding = FourierEmb(pos_dim)
        self.dropout = dropout

    def forward(self, meg, positions):
        scores = self.embedding(positions) @ self.heads[0].T
        if self.training and self.dropout:
            centre = torch.rand(2, device=meg.device)
            excluded = (positions - centre).norm(dim=-1) <= self.dropout
            scores = scores.masked_fill(excluded[:, None], float("-inf"))
        weights = torch.softmax(scores.float(), 0).to(meg.dtype)
        return torch.einsum("bct,co->bot", meg, weights)


class ConvBlock(nn.Module):
    def __init__(self, chin, chout, dilation, last=False, glu=False, drop_in=0.0):
        super().__init__()
        layers = []
        if drop_in:
            layers.append(nn.Dropout(drop_in))
        layers.append(nn.Conv1d(chin, chout, 3, 1, dilation, dilation=dilation))
        if not last:
            layers.extend([nn.BatchNorm1d(chout), nn.GELU()])
        self.block = nn.Sequential(*layers)
        self.glu = None
        if glu:
            self.glu = nn.Sequential(
                nn.Conv1d(chout, 2 * chout, 3, padding=1), nn.GLU(dim=1)
            )

    def forward(self, x):
        output = self.block(x)
        if output.shape == x.shape:
            output = output + x
        if self.glu is not None:
            output = self.glu(output)
        return output


class BrainModule(nn.Module):
    def __init__(self, out_dim=1024, hidden=160, depth=5, initial=512, agg_hidden=256):
        super().__init__()
        self.merger = ChannelMerger()
        self.initial = nn.Conv1d(270, initial, 1)
        self.subject = nn.Conv1d(initial, initial, 1, bias=False)
        sizes = [initial] + [hidden] * (depth - 1) + [out_dim]
        blocks = []
        for layer in range(depth):
            blocks.append(
                ConvBlock(
                    sizes[layer],
                    sizes[layer + 1],
                    2**layer,
                    last=layer == depth - 1,
                    glu=(layer + 1) % 2 == 0,
                    drop_in=0.1 if layer == 0 else 0.0,
                )
            )
        self.encoder = nn.Sequential(*blocks)
        self.att = nn.Sequential(
            nn.Conv1d(out_dim, agg_hidden, 1), nn.Tanh(), nn.Conv1d(agg_hidden, 1, 1)
        )

    def forward(self, x, positions):
        x = self.merger(x, positions)
        x = self.initial(x)
        x = self.subject(x)
        x = self.encoder(x)
        attention = torch.softmax(self.att(x), dim=-1)
        return (x * attention).sum(-1)


class Decoder(nn.Module):

    def __init__(self, dim=1024):
        super().__init__()
        self.cnn = BrainModule(out_dim=dim)
        self.cls = nn.Linear(dim, 51)
        self.transformer = Encoder(
            dim=dim,
            depth=DEPTH,
            heads=16,
            rotary_pos_emb=True,
            attn_dropout=0.1,
            ff_dropout=0.0,
            use_scalenorm=True,
            scale_residual=True,
        )
        self.logit_scale = nn.Parameter(torch.tensor(math.log(10.0)))
        self.bias = nn.Parameter(torch.tensor(-10.0))

    def forward(self, real, positions, mask):
        emb = F.normalize(self.cnn(real, positions), dim=-1)
        h = emb.new_zeros(*mask.shape, emb.shape[-1])
        h = h.masked_scatter(mask[..., None], emb)
        return F.normalize(self.transformer(h, mask=mask), dim=-1)[mask]

In [ ]:
NETS = {}
for run, path in CHECKPOINTS.items():
    checkpoint = torch.load(path, map_location="cpu")
    cfg = checkpoint["config"]
    assert cfg["run"] == run and cfg["cache"] == "d2"
    assert cfg["duration"] == 2.5 and cfg["clamp"] == 5.0
    assert cfg["ce_weight"] == 1.0
    net = Decoder()
    net.load_state_dict(checkpoint["state_dict"], strict=True)
    NETS[run] = (net.to(device).eval(), cfg)
    print(f"{path.name}: {cfg}")
DURATION = 2.5
CLAMP = 5.0
window_samples = int(round(DURATION * FREQ))
vocabulary = load_vocabulary("primary")


def normalise_word(w):
    return "".join(
        (c for c in str(w).replace("’", "'").lower() if c.isalnum() or c in "-'")
    )


word_row = {str(w): i for i, w in enumerate(metadata["words"])}
candidate_embeddings = F.normalize(
    torch.tensor(
        np.asarray(
            word_embeddings[[word_row[normalise_word(v)] for v in vocabulary]],
            dtype=np.float32,
        )
    ),
    dim=-1,
).to(device)

In [ ]:
def raw_windows(store, starts, rooms):
    windows = np.zeros((len(starts), window_samples, 306), dtype=np.float16)
    for row, (start, available) in enumerate(zip(starts, rooms)):
        length = min(window_samples, int(available))
        windows[row, :length] = store[start : start + length]
    return windows


def scale_windows(raw, rooms, chunk=256):
    batches = []
    for windows, lengths in zip(raw.float().split(chunk), rooms.split(chunk)):
        valid = (
            torch.arange(window_samples, device=windows.device)[None] < lengths[:, None]
        )
        valid = valid[..., None]
        quantiles = torch.nanquantile(
            windows.masked_fill(~valid, float("nan")),
            torch.tensor([0.25, 0.5, 0.75], device=windows.device),
            dim=1,
        )
        median = quantiles[1][:, None]
        spread = (quantiles[2] - quantiles[0]).clamp_min(1e-6)[:, None]
        windows = (windows - median) / spread
        windows = windows.clamp(-CLAMP, CLAMP) * valid
        batches.append(windows.transpose(1, 2))
    return torch.cat(batches)


class Groups(torch.utils.data.Dataset):
    def __init__(self, store, at, groups, rooms):
        self.store = store
        self.at = at
        self.groups = groups
        self.rooms = rooms

    def __len__(self):
        return len(self.groups)

    def __getitem__(self, index):
        rows = self.groups[index]
        windows = raw_windows(self.store, self.at[rows], self.rooms[index])
        lengths = np.asarray(self.rooms[index], dtype=np.int64)
        return torch.from_numpy(windows), torch.from_numpy(lengths)


def collate(items):
    batch_size = len(items)
    max_words = max(len(item[0]) for item in items)
    raw = torch.zeros(batch_size, max_words, window_samples, 306, dtype=torch.float16)
    rooms = torch.ones(batch_size, max_words, dtype=torch.long)
    mask = torch.zeros(batch_size, max_words, dtype=torch.bool)
    for row, (windows, lengths) in enumerate(items):
        count = len(windows)
        raw[row, :count] = windows
        rooms[row, :count] = lengths
        mask[row, :count] = True
    return raw, rooms, mask


def batches(store, at, groups, rooms):
    per = max(1, int(256 / np.mean([len(g) for g in groups])))
    return DataLoader(
        Groups(store, at, groups, rooms),
        batch_size=per,
        collate_fn=collate,
        num_workers=NUM_WORKERS,
        pin_memory=device == "cuda",
    )


h_epoch, h_at = (
    metadata["hold_epoch"].astype(np.int64),
    metadata["hold_at"].astype(np.int64),
)
h_start, h_len = (
    metadata["hold_start"].astype(np.int64),
    metadata["hold_len"].astype(np.int64),
)
_o = np.argsort(h_epoch, kind="stable")
HOLD_GROUPS = np.split(_o, np.flatnonzero(np.diff(h_epoch[_o])) + 1)
HOLD_ROOM = h_start[h_epoch] + h_len[h_epoch] - h_at
for g in HOLD_GROUPS:
    assert len(set(h_subject[g].tolist())) == 1 and (np.diff(h_at[g]) >= 0).all()


def zrow(s):
    return (s - s.mean(1, keepdims=True)) / np.maximum(s.std(1, keepdims=True), 1e-06)


@torch.inference_mode()
def score(net, store, at, groups, rooms):
    net.eval()
    retrieval_scores = []
    classifier_scores = []
    for raw, lengths, mask in batches(store, at, groups, rooms):
        raw = raw.to(device)
        lengths = lengths.to(device)
        mask = mask.to(device)
        windows = scale_windows(raw[mask], lengths[mask])
        with torch.autocast(**autocast_settings):
            embeddings = net(windows, sensor_positions, mask).float()
        retrieval_scores.append((embeddings @ candidate_embeddings.T).cpu())
        classifier_scores.append(net.cls(embeddings)[:, :50].float().cpu())
    retrieval_scores = torch.cat(retrieval_scores).numpy()
    classifier_scores = torch.cat(classifier_scores).numpy()
    return zrow(retrieval_scores) + zrow(classifier_scores)

In [ ]:
hold_subjects = sorted(set(h_subject.tolist()))
HOLD = {s: [g for g in HOLD_GROUPS if h_subject[g[0]] == s] for s in hold_subjects}
members = list(NETS)
HOLD_SCORES = {}
for name, (net, cfg) in NETS.items():
    out = np.zeros((len(h_subject), 50), dtype=np.float32)
    for s, groups in HOLD.items():
        rows = np.concatenate(groups)
        out[rows] = score(
            net, holdout_meg, h_at, groups, [HOLD_ROOM[g] for g in groups]
        )
    HOLD_SCORES[name] = out
    np.save(OUTPUT_DIR / f"broad_holdout_scores_b0_{name}_plain.npy", out)
    print(f"{name}: holdout served, {(time.time() - T0) / 60:.1f} min", flush=True)
ens = np.mean([zrow(HOLD_SCORES[m]) for m in members], axis=0)
np.save(OUTPUT_DIR / "broad_holdout_scores_b0_ens_plain.npy", ens)
ranks = ens.argsort(1).argsort(1).astype(np.float64) + 1
SUB = OUTPUT_DIR / "submission_broad.csv"
write_submission(
    str(SUB),
    indices=metadata["hold_indices"],
    primary_probs=ranks / ranks.sum(1, keepdims=True),
)
print(f"wrote {SUB.name}")